# 04: PII 保護・遡及削除（GDPR/CCPA）とタイムトラベル

## このノートブックで学習できること

- パターン 1（取り込み時分離 - PII Vault）: `crm_customers` 取り込み時に生 PII（氏名・メール・電話番号）を `pii_vault` へ分離し、通常の Bronze テーブルには SHA-256 ハッシュ化した値のみを保存する実装
- パターン 2（GDPR 遡及マスクと物理抹消）: Day 2 で `status = 'WITHDRAWN'` となった退会顧客について PII を `'***REDACTED***'` にマスクし、タイムトラベルで旧スナップショットに残る PII を確認したうえで `expire_snapshots` / `vacuum`（保持期間 0 指定）により物理ファイルごと完全抹消する手順
- カタログ層（BigLake ポリシータグ / Unity Catalog Column Mask）による動的データマスキングの仕組みと、マルチエンジン環境での注意点

## 到達ゴール

- 通常の `UPDATE` や `DELETE` だけでは過去スナップショットに個人情報が残り続ける理由を説明でき、スナップショット失効と物理ファイル削除まで含めた完全抹消を実行できる

## なぜこれが必要か（従来のデータレイクとの比較）

データレイクでは追記や上書きを行っても過去ファイルが残りやすく、退会者の削除要求（GDPR の忘れられる権利など）に対して「どこにどのファイルが残っているか」を追跡するのが困難でした。PII Vault への分離と、OTF のスナップショット失効・物理削除を組み合わせることで、分析用テーブルの履歴を保ちながら法的な抹消要件を満たせます。


## 1. パターン 1: PII Vault 分離とハッシュ化 Bronze テーブルの作成

このセルでは、Day 1 の `crm_customers.csv` から生 PII（`name`, `email`, `phone`）を切り出して `pii_vault.customers_pii` に保存し、通常の分析用テーブル `bronze.crm_customers_hashed` には SHA-256 ハッシュ値だけを保存します。


In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.catalog_helper import (
    LANDING_DIR,
    LAKEHOUSE_DIR,
    get_duckdb_conn,
    get_iceberg_catalog,
    init_storage,
    register_iceberg_table,
)
import hashlib
import json
import polars as pl
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake

catalog = get_iceberg_catalog()
for tbl_name in ["pii_vault.customers_pii", "bronze.crm_customers_hashed"]:
    if catalog.table_exists(tbl_name):
        catalog.drop_table(tbl_name)

def sha256_hex(val: str) -> str:
    return hashlib.sha256(val.encode("utf-8")).hexdigest()[:16]

d1_cust = pl.read_csv(LANDING_DIR / "day1" / "crm_customers.csv")

pii_d1 = d1_cust.select(["customer_id", "name", "email", "phone", "birth_date", "status", "updated_at"])
hashed_d1 = d1_cust.with_columns([
    pl.col("name").map_elements(sha256_hex, return_dtype=pl.String).alias("name_hash"),
    pl.col("email").map_elements(sha256_hex, return_dtype=pl.String).alias("email_hash"),
    pl.col("phone").map_elements(sha256_hex, return_dtype=pl.String).alias("phone_hash"),
]).select(["customer_id", "name_hash", "email_hash", "phone_hash", "membership_tier", "status", "updated_at"])

pii_tbl = catalog.create_table("pii_vault.customers_pii", schema=pii_d1.to_arrow().schema)
pii_tbl.append(pii_d1.to_arrow())

hashed_tbl = catalog.create_table("bronze.crm_customers_hashed", schema=hashed_d1.to_arrow().schema)
hashed_tbl.append(hashed_d1.to_arrow())

# Delta Lake 版 PII Vault も作成（Deletion Vector / UPDATE / VACUUM 比較用）
delta_pii_path = LAKEHOUSE_DIR / "pii_vault" / "customers_pii_delta"
write_deltalake(str(delta_pii_path), pii_d1.to_arrow(), mode="overwrite")

print("Bronze (ハッシュ済み) 先頭3件:")
print(pl.from_arrow(hashed_tbl.scan().to_arrow()).head(3))


Bronze テーブルには `name_hash`・`email_hash`・`phone_hash` のみが保存され、生 PII は `pii_vault` に隔離されました。

## 2. パターン 2: 退会顧客の遡及マスクとタイムトラベル確認

次のセルでは、Day 2 の `crm_customers.csv` で `status == 'WITHDRAWN'` となった 10 名を特定し、Iceberg の `pii_vault.customers_pii` と Delta Lake の `customers_pii_delta` の両方に対して PII を `'***REDACTED***'` にマスクする更新を実行します。その後、マスク前の過去スナップショットへ Time Travel すると元の PII がまだ参照できてしまうことを確認します。


In [ ]:
d2_cust = pl.read_csv(LANDING_DIR / "day2" / "crm_customers.csv")
withdrawn_ids = d2_cust.filter(pl.col("status") == "WITHDRAWN")["customer_id"].to_list()
print(f"Day 2 退会顧客数: {len(withdrawn_ids)} 件 (例: {withdrawn_ids[:3]})")

snap_before_mask = pii_tbl.current_snapshot().snapshot_id
delta_dt = DeltaTable(str(delta_pii_path))
delta_ver_before = delta_dt.version()

# Iceberg 側のマスク更新（Copy-on-Write で新スナップショット作成）
cur_pii = pl.from_arrow(pii_tbl.scan().to_arrow())
masked_pii = cur_pii.with_columns([
    pl.when(pl.col("customer_id").is_in(withdrawn_ids)).then(pl.lit("***REDACTED***")).otherwise(pl.col("name")).alias("name"),
    pl.when(pl.col("customer_id").is_in(withdrawn_ids)).then(pl.lit("***REDACTED***")).otherwise(pl.col("email")).alias("email"),
    pl.when(pl.col("customer_id").is_in(withdrawn_ids)).then(pl.lit("***REDACTED***")).otherwise(pl.col("phone")).alias("phone"),
    pl.when(pl.col("customer_id").is_in(withdrawn_ids)).then(pl.lit("WITHDRAWN")).otherwise(pl.col("status")).alias("status"),
])
pii_tbl.overwrite(masked_pii.to_arrow())

# Delta Lake 側の直接 UPDATE
id_list_sql = ", ".join(f"'{cid}'" for cid in withdrawn_ids)
delta_dt.update(
    predicate=f"customer_id IN ({id_list_sql})",
    updates={"name": "'***REDACTED***'", "email": "'***REDACTED***'", "phone": "'***REDACTED***'", "status": "'WITHDRAWN'"},
)

print("最新 Iceberg での退会顧客:")
print(pl.from_arrow(pii_tbl.scan().to_arrow()).filter(pl.col("customer_id") == withdrawn_ids[0]))
print("マスク前 Iceberg スナップショット (Time Travel) での同一顧客:")
print(pl.from_arrow(pii_tbl.scan(snapshot_id=snap_before_mask).to_arrow()).filter(pl.col("customer_id") == withdrawn_ids[0]))


最新スナップショットでは `***REDACTED***` になりましたが、`snapshot_id=snap_before_mask` へ Time Travel すると退会顧客の氏名・メールアドレス・電話番号がそのまま読めてしまいます。法的な抹消要件を満たすには、過去スナップショットの失効と物理ファイルの削除が必要です。

## 3. 物理抹消（Expire Snapshots / Vacuum）と削除の実証

次のセルでは、Iceberg 側で履歴保持期間を最小にして `expire_snapshots` を実行したうえで未参照の旧 Parquet ファイルを物理削除し、Delta Lake 側でも `vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=False)` を実行します。その後、過去の Time Travel 参照が失敗（物理的に消去）することを実証します。


In [ ]:
from datetime import datetime, timezone

# Iceberg: 履歴保持プロパティを短くして旧スナップショットを失効させ、未参照ファイルを削除
with pii_tbl.transaction() as tx:
    tx.set_properties({"history.expire.min-snapshots-to-keep": "1", "history.expire.max-snapshot-age-ms": "0"})
cur_ts = datetime.fromtimestamp(pii_tbl.current_snapshot().timestamp_ms / 1000, tz=timezone.utc)
pii_tbl.maintenance.expire_snapshots().older_than(cur_ts).commit()

cur_paths = {Path(t.file.file_path.replace("file://", "")).resolve() for t in pii_tbl.scan().plan_files()}
pii_data_dir = Path(pii_tbl.location().replace("file://", "")) / "data"
removed_ice = 0
for pq in pii_data_dir.rglob("*.parquet"):
    if pq.resolve() not in cur_paths:
        pq.unlink()
        removed_ice += 1

# Delta Lake: retention_hours=0 で VACUUM 実行
removed_delta = delta_dt.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=False)
print(f"Iceberg 物理削除ファイル数: {removed_ice} / Delta VACUUM 削除数: {len(removed_delta)}")

for label, fn in [
    ("Iceberg Time Travel", lambda: pii_tbl.scan(snapshot_id=snap_before_mask).to_arrow()),
    ("Delta Time Travel", lambda: DeltaTable(str(delta_pii_path), version=delta_ver_before).to_pyarrow_table()),
]:
    try:
        fn()
        print(f"{label}: 参照可能")
    except Exception as e:
        print(f"{label}: 参照不可 ({type(e).__name__})")


Iceberg・Delta Lake ともに旧バージョンの物理 Parquet ファイルが削除され、マスク前の Time Travel 参照がエラーになって物理抹消が完了しました。

## 4. 動的データマスキング（Dynamic Data Masking）とマルチエンジンの注意点

- カタログ層の動的マスキング: BigQuery（BigLake ポリシータグ）や Databricks（Unity Catalog の Column Mask）では、テーブルの物理データを書き換えず、参照するユーザーのロールに応じてクエリ実行時に `***MASKED***` へ変換して返します。
- マルチエンジン環境での注意点: BigQuery で設定したポリシータグは BigQuery 経由のクエリには効きますが、同一 GCS 上の Iceberg ファイルを Databricks や Spark が直接読む場合は BigQuery のポリシーが自動適用されません。そのため、全社共通で隠蔽すべき PII は本ノートブック前半のように取り込み時点で PII Vault へ分離・ハッシュ化し、物理的にアクセス可能なバケット/テーブル自体を分ける設計が安全です。
